In [ ]:
#Importin the necessary libraries
import pyspark
from pyspark import SparkContext

sc= SparkContext()
spark= pyspark.sql.SparkSession(sc,jsparkSession=None)

## Raw read file

With this option, you get to read both the headers and infer the schema so it can recognize the header and the data types

In [ ]:
CarDf=spark.read.option("header","true").option("inferSchema","true").csv("car_prices.csv")

In [ ]:
#print Schema
CarDf.printSchema()

In [ ]:
#show the content of the file
CarDf.show()

## Creating A manually built schema
Optionally we will use our own data type so it wont get it wrong

In [ ]:
from pyspark.sql.types import *

In [ ]:
columns=[StructField("year",IntegerType()),
         StructField("make",StringType()),
         StructField("model",StringType()),
         StructField("trim",StringType()),
         StructField("body",StringType()),
         StructField("transmission",StringType()),
         StructField("vin",StringType()),
         StructField("state",StringType()),
         StructField("condition",IntegerType()),
         StructField("odometer",IntegerType()),
         StructField("color",StringType()),
         StructField("interior",StringType()),
         StructField("seller",StringType()),
         StructField("sellingprice",IntegerType()),
         StructField("mmr",IntegerType()),
         StructField("saledate",DateType())]
carSchema=StructType(columns)

In [ ]:
ManualSchema=spark.read.schema(carSchema).csv("car_prices.csv")

In [ ]:
ManualSchema.printSchema()

In [ ]:
#to show the columns
ManualSchema.columns().show()

ManualSchema.show()

## Checking for Kia and Audi Cars

In [ ]:
Kia_And_Audi=ManualSchema.where("make ==Kia or make==Audi")

In [ ]:
Kia_And_Audi.show()

In [ ]:
#to join files
#Kia_And_Audi.join(Manual_Schema,"vin").show()

Accessing the model column

In [ ]:
ManualSchema.select("model").show

to show the make and model of each car

In [ ]:
ManualSchema.select(ManualSchema.make,"model").show()

To convert the selling price to Naira

In [ ]:
ManualSchema.select("make","model","trim",(ManualSchema.sellingprice*2400).alias("sellingprice_naira")).show()

checking for all "series" cars

In [ ]:
ManualSchema.where(ManualSchema.model.contains("Series")).show()

How is the make distributed

In [ ]:
ManualSchema.groupBy(ManualSchema.make).count().show()

# EDA

In [ ]:
# creating a temporary view of the cars
ManualSchema.createOrReplaceTempView("TempView")

# viewing those with manual transmissions
spark.sql("Select * from TempView where transmission = manual").show()

to list all the tables available

In [ ]:
spark.catalog.listTables()

# Creating RDDs
In Spark, RDDs (Resilient Distributed Datasets) are the fundamental data structure, representing an immutable, partitioned collection of data that can be processed in parallel across a cluster

In [ ]:
#creating an rdd of the cars
carsrdd=ManualSchema.rdd

In [ ]:
#to chcek the rows
for row in carsrdd.take(10):
    print(row)

In [ ]:
#to turn it back into a dataframe
RevertedDF=spark.createDataFrame(carsrdd,columns)

RevertedDF.show()

## Advanced EDA Steps

In [ ]:
# Drop rows where price is missing
CarDf = CarDf.na.drop(subset=["price"])
print("Missing values in 'price' column handled.")

In [ ]:
from pyspark.sql.functions import col

# Cast price to float
CarDf = CarDf.withColumn("price", col("price").cast("float"))
print("Price column casted to float.")

In [ ]:
from pyspark.sql.functions import approxQuantile

# Calculate IQR for price
q1, q3 = CarDf.approxQuantile("price", [0.25, 0.75], 0.0)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Filter out outliers
CarDf = CarDf.filter((col("price") >= lower_bound) & (col("price") <= upper_bound))
print(f"Outliers removed using IQR. Lower: {lower_bound}, Upper: {upper_bound}")

In [ ]:
# Aggregating average price by model
CarDf.groupBy("model").agg({"price": "avg"}).show()

In [ ]:
# Example correlation between price and odometer, adjust as needed
if 'odometer' in CarDf.columns:
    CarDf = CarDf.withColumn("odometer", col("odometer").cast("float"))
    correlation = CarDf.stat.corr("price", "odometer")
    print(f"Correlation between price and odometer: {correlation}")
else:
    print("Odometer column not found for correlation analysis.")

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

# Ranking cars by price
windowSpec = Window.orderBy(col("price").desc())
ranked_df = CarDf.withColumn("rank", rank().over(windowSpec))
ranked_df.select("model", "price", "rank").show(10)

In [ ]:
# Save cleaned data to CSV
CarDf.write.csv("cleaned_car_prices.csv", header=True, mode="overwrite")
print("Cleaned data saved to 'cleaned_car_prices.csv'")